In [2]:
import aria2p
from pathlib import Path
import time


def download_torrents(torrent_folder, output_dir='./downloads', max_connections=5, seed_time=0):
    """
    Main function to download all torrents from a folder using aria2p.

    Args:
        torrent_folder: Path to folder containing .torrent files
        output_dir: Directory where to save downloaded content (default: ./downloads)
        max_connections: Maximum connections per server (default: 5)
        seed_time: Seed time in minutes (default: 0, no seeding)
    """
    try:
        # Start aria2c daemon and connect to it
        aria2 = aria2p.API(
            aria2p.Client(
                host="http://localhost",
                port=6800,
                secret=""
            )
        )

        # Test connection
        aria2.get_stats()
        print("✓ Connected to aria2c daemon")

    except Exception as e:
        print("Starting aria2c daemon...")
        import subprocess
        import os

        # Start aria2c as a daemon
        daemon_process = subprocess.Popen(
            ['aria2c', '--enable-rpc', '--rpc-listen-all=false', '--rpc-listen-port=6800'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        # Wait for daemon to start
        time.sleep(2)

        try:
            aria2 = aria2p.API(
                aria2p.Client(
                    host="http://localhost",
                    port=6800,
                    secret=""
                )
            )
            print("✓ Started and connected to aria2c daemon")
        except Exception as e:
            print(f"Error: Could not connect to aria2c daemon: {e}")
            print("Make sure aria2c is installed:")
            print("  Ubuntu/Debian: sudo apt install aria2")
            print("  macOS: brew install aria2")
            return

    # Validate torrent folder
    torrent_folder = Path(torrent_folder)
    if not torrent_folder.exists():
        print(f"Error: Torrent folder '{torrent_folder}' does not exist")
        return

    if not torrent_folder.is_dir():
        print(f"Error: '{torrent_folder}' is not a directory")
        return

    # Create output directory
    output_dir = Path(output_dir).absolute()
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {output_dir}")

    # Find all .torrent files
    torrent_files = list(torrent_folder.glob('*.torrent'))

    if not torrent_files:
        print(f"No .torrent files found in {torrent_folder}")
        return

    print(f"\nFound {len(torrent_files)} torrent file(s)")
    print(f"{'='*60}\n")

    # Add all torrents to aria2
    downloads = []
    for torrent_file in torrent_files:
        try:
            options = {
                'dir': str(output_dir),
                'max-connection-per-server': str(max_connections),
                'seed-time': str(seed_time),
                'continue': 'true',
            }

            download = aria2.add_torrent(str(torrent_file), options=options)
            downloads.append(download)
            print(f"✓ Added: {torrent_file.name}")

        except Exception as e:
            print(f"✗ Error adding {torrent_file.name}: {e}")

    print(f"\n{'='*60}")
    print(f"Downloading {len(downloads)} torrent(s)...")
    print(f"{'='*60}\n")

    # Monitor downloads
    try:
        while True:
            # Refresh download status
            active_downloads = []
            completed = 0

            for download in downloads:
                download.update()

                if download.is_complete:
                    completed += 1
                elif not download.has_failed:
                    active_downloads.append(download)

            # Display progress
            print(f"\rProgress: {completed}/{len(downloads)} completed", end='', flush=True)

            # Show active download details
            if active_downloads:
                print("\n")
                for dl in active_downloads[:3]:  # Show first 3 active downloads
                    progress = dl.progress
                    speed = dl.download_speed_string()
                    name = dl.name[:50] + "..." if len(dl.name) > 50 else dl.name
                    print(f"  {name}: {progress:.1f}% @ {speed}")
                print()

            # Check if all downloads are complete or failed
            if completed == len(downloads):
                print(f"\n\n{'='*60}")
                print("All downloads completed!")
                break

            # Check for failures
            failed = [d for d in downloads if d.has_failed]
            if completed + len(failed) == len(downloads):
                print(f"\n\n{'='*60}")
                print("Downloads finished (some failed)")
                break

            time.sleep(2)

    except KeyboardInterrupt:
        print("\n\nDownload interrupted by user")

    # Summary
    successful = sum(1 for d in downloads if d.is_complete)
    failed = sum(1 for d in downloads if d.has_failed)

    print(f"{'='*60}")
    print("DOWNLOAD SUMMARY")
    print(f"{'='*60}")
    print(f"Total torrents: {len(downloads)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Output location: {output_dir}")
    print(f"{'='*60}")

In [3]:
download_torrents("./torrent-files", "./zstandard_files")

Starting aria2c daemon...


FileNotFoundError: [Errno 2] No such file or directory: 'aria2c'